# **EMAIL SPAM DETECTION MODEL**


# **Step#1 Import Libraries**

In [8]:


import numpy as np
import pandas as pd
import pickle
import zipfile
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import MaxAbsScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
import warnings
warnings.filterwarnings('ignore')

# **Step#2 Load Data**

In [9]:
try:
    with zipfile.ZipFile("/content/emails.csv.zip") as z:
        csv_name = [f for f in z.namelist() if f.endswith('.csv')][0]
        with z.open(csv_name) as f:
            df = pd.read_csv(f)
except Exception:
    df = pd.read_csv("/content/emails.csv.zip")

print(f"Dataset shape: {df.shape}")
print(f"Columns (first 5): {df.columns[:5].tolist()} ...")
print(f"\nLabel distribution:\n{df['Prediction'].value_counts()}")


Dataset shape: (5172, 3002)
Columns (first 5): ['Email No.', 'the', 'to', 'ect', 'and'] ...

Label distribution:
Prediction
0    3672
1    1500
Name: count, dtype: int64


# **Step#3 PREPARE FEATURES & LABELS**

In [10]:
# Drop non-feature columns
drop_cols = ['Email No.', 'Prediction']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].values   # word-count matrix (already numeric)
y = df['Prediction'].values   # 0 = ham, 1 = spam

print(f"\nFeature matrix shape : {X.shape}")
print(f"Spam emails          : {y.sum()}  ({y.mean()*100:.1f}%)")
print(f"Ham  emails          : {(y==0).sum()}  ({(y==0).mean()*100:.1f}%)")


Feature matrix shape : (5172, 3000)
Spam emails          : 1500  (29.0%)
Ham  emails          : 3672  (71.0%)


# **Step#4 Train-Test Split**

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")



Train: 4137  |  Test: 1035


# **Step#5 Scaling**

In [12]:
scaler = MaxAbsScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)


# **Step#6 Define Models**

In [13]:
models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced', random_state=42),
    'Naive Bayes': MultinomialNB(alpha=0.5),          # needs non-negative X
    'Random Forest': RandomForestClassifier(
        n_estimators=200, n_jobs=-1,
        class_weight='balanced', random_state=42),
}


# **Step#7 Model Evaluation**

In [14]:

best_model, best_auc, best_name = None, 0.0, ''

for name, clf in models.items():
    # Naive Bayes needs raw counts (non-negative), others use scaled
    X_tr = X_train if name == 'Naive Bayes' else X_train_s
    X_te = X_test  if name == 'Naive Bayes' else X_test_s

    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    auc    = roc_auc_score(y_test, y_prob)

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"  Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"  ROC-AUC  : {auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Ham','Spam']))

    if auc > best_auc:
        best_auc, best_model, best_name = auc, clf, name

print(f"\nBest individual model: {best_name}  (AUC={best_auc:.4f})")


  Logistic Regression
  Accuracy : 0.9565
  ROC-AUC  : 0.9897
              precision    recall  f1-score   support

         Ham       0.99      0.95      0.97       735
        Spam       0.88      0.98      0.93       300

    accuracy                           0.96      1035
   macro avg       0.94      0.96      0.95      1035
weighted avg       0.96      0.96      0.96      1035


  Naive Bayes
  Accuracy : 0.9440
  ROC-AUC  : 0.9751
              precision    recall  f1-score   support

         Ham       0.98      0.94      0.96       735
        Spam       0.87      0.94      0.91       300

    accuracy                           0.94      1035
   macro avg       0.92      0.94      0.93      1035
weighted avg       0.95      0.94      0.94      1035


  Random Forest
  Accuracy : 0.9739
  ROC-AUC  : 0.9966
              precision    recall  f1-score   support

         Ham       0.99      0.98      0.98       735
        Spam       0.94      0.97      0.96       300

    acc

# **Step#8 Voting ensemble**

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler as MAS

# Wrap each in its own pipeline so scaling is handled internally
pipe_lr = Pipeline([('scaler', MAS()),
                    ('clf', LogisticRegression(C=1.0, max_iter=1000,
                                               class_weight='balanced',
                                               random_state=42))])
pipe_nb = Pipeline([('clf', MultinomialNB(alpha=0.5))])   # raw counts
pipe_rf = Pipeline([('scaler', MAS()),
                    ('clf', RandomForestClassifier(n_estimators=200,
                                                   n_jobs=-1,
                                                   class_weight='balanced',
                                                   random_state=42))])

ensemble = VotingClassifier(
    estimators=[('lr', pipe_lr), ('nb', pipe_nb), ('rf', pipe_rf)],
    voting='soft'
)
ensemble.fit(X_train, y_train)
y_pred_ens = ensemble.predict(X_test)
y_prob_ens = ensemble.predict_proba(X_test)[:, 1]
ens_auc    = roc_auc_score(y_test, y_prob_ens)

print(f"\n{'='*50}")
print("  Voting Ensemble (LR + NB + RF)")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_ens):.4f}")
print(f"  ROC-AUC  : {ens_auc:.4f}")
print(classification_report(y_test, y_pred_ens, target_names=['Ham','Spam']))
print(f"  Confusion Matrix:\n{confusion_matrix(y_test, y_pred_ens)}")


  Voting Ensemble (LR + NB + RF)
  Accuracy : 0.9691
  ROC-AUC  : 0.9960
              precision    recall  f1-score   support

         Ham       0.99      0.97      0.98       735
        Spam       0.92      0.97      0.95       300

    accuracy                           0.97      1035
   macro avg       0.96      0.97      0.96      1035
weighted avg       0.97      0.97      0.97      1035

  Confusion Matrix:
[[711  24]
 [  8 292]]


# **Step#9 Best Model**

In [16]:
if ens_auc >= best_auc:
    final_model   = ensemble
    final_name    = 'Voting Ensemble'
    uses_raw      = True     # ensemble pipelines handle scaling internally
else:
    final_model   = best_model
    final_name    = best_name
    uses_raw      = (best_name == 'Naive Bayes')

print(f"\n Final model selected: {final_name}")


 Final model selected: Random Forest


# **Step#10 Save pickele**

In [17]:
model_bundle = {
    'model':         final_model,
    'scaler':        scaler,          # MaxAbsScaler fitted on train set
    'feature_cols':  feature_cols,    # list of 3000 word columns
    'label_map':     {0: 'ham', 1: 'spam'},
    'model_name':    final_name,
    'uses_raw_counts': uses_raw,      # True = pass raw counts; False = scale first
}

SAVE_PATH = "/content/spam_detector.pkl"
with open(SAVE_PATH, 'wb') as f:
    pickle.dump(model_bundle, f)

print(f"\nModel saved → {SAVE_PATH}")



Model saved → /content/spam_detector.pkl


# **Step#10 Verification**

In [18]:
with open(SAVE_PATH, 'rb') as f:
    loaded = pickle.load(f)

def predict_email(word_count_dict: dict) -> dict:
    """
    word_count_dict: {word: count, ...}  e.g. {'free': 3, 'click': 2}
    Returns: {'label': 'spam'/'ham', 'spam_probability': float}
    """
    cols   = loaded['feature_cols']
    vec    = np.array([[word_count_dict.get(w, 0) for w in cols]])
    model  = loaded['model']

    if loaded['uses_raw_counts']:
        X_in = vec
    else:
        X_in = loaded['scaler'].transform(vec)

    label = model.predict(X_in)[0]
    prob  = model.predict_proba(X_in)[0][1]
    return {'label': loaded['label_map'][label],
            'spam_probability': round(float(prob), 4)}

# Test with a sample row from the dataset
sample_spam = dict(zip(feature_cols, X_test[y_test == 1][0]))
sample_ham  = dict(zip(feature_cols, X_test[y_test == 0][0]))

print("\n── Live Predictions ────────────────────────────────────")
r1 = predict_email(sample_spam)
r2 = predict_email(sample_ham)
icon1 = " SPAM" if r1['label'] == 'spam' else " HAM"
icon2 = " SPAM" if r2['label'] == 'spam' else " HAM"
print(f"Known SPAM email → {icon1}  (prob={r1['spam_probability']:.2%})")
print(f"Known HAM  email → {icon2}  (prob={r2['spam_probability']:.2%})")

print(f"\n All done! Pickle saved at: {SAVE_PATH}")


── Live Predictions ────────────────────────────────────
Known SPAM email →  SPAM  (prob=65.50%)
Known HAM  email →  HAM  (prob=17.00%)

 All done! Pickle saved at: /content/spam_detector.pkl


# **Removing Scalar dependency**

In [19]:
# Run in Colab — re-save the pickle with scaler removed
import pickle

with open("/content/spam_detector.pkl", "rb") as f:
    bundle = pickle.load(f)

# Ensemble pipelines handle scaling internally — drop standalone scaler
bundle['scaler'] = None
bundle['uses_raw_counts'] = True   # always pass raw counts to the bundle model

with open("/content/spam_detector.pkl", "wb") as f:
    pickle.dump(bundle, f)

print("✅ Re-saved without scaler dependency")

from google.colab import files
files.download("/content/spam_detector.pkl")

✅ Re-saved without scaler dependency


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>